# Odchylka ČR: z prvních 5 minut odhad dalších 10 minut

Spusťte buňky postupně shora dolů. Notebook stáhne celý skript z GitHubu;
není potřeba kopírovat dlouhý kód. Připravte alespoň čtyři dny minutových CSV.

Příklad: vstup `12:00–12:04`, cíl `12:05–12:14`. Vyhodnocení vždy používá
jen starší dny k učení. Správná polarita znamená znaménko **průměru** dalších
deseti minut, nikoli stejnou polaritu po celých deset minut.


## 1. Nainstalovat závislosti


In [ ]:
%pip -q install pandas numpy matplotlib scikit-learn
%matplotlib inline


## 2. Stáhnout úplný skript


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

script = Path("predikce_ctvrthodin.py")
urlretrieve(
    "https://raw.githubusercontent.com/000307/Odchylky/main/predikce_ctvrthodin.py",
    script,
)
# Zkontrolovat, že se stáhl syntakticky úplný Python soubor.
compile(script.read_text(encoding="utf-8"), str(script), "exec")
print(f"Stažen úplný skript: {script} ({script.stat().st_size} bajtů)")


## 3. Nahrát všechna vstupní CSV


In [ ]:
from google.colab import files

uploaded = files.upload()
csv_files = sorted(name for name in uploaded if name.lower().endswith(".csv"))
if not csv_files:
    raise ValueError("Nahrajte vstupní CSV soubory s minutovými odchylkami.")
print(f"Vybráno CSV souborů: {len(csv_files)}")


## 4. Spustit časové vyhodnocení a zobrazit graf


In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location("predikce_ctvrthodin", script)
predikce = importlib.util.module_from_spec(spec)
spec.loader.exec_module(predikce)
# Explicitní argumenty nepřebírají interní parametry kernelu (např. -f).
predikce.main([*csv_files, "--out", "predikce"])


## 5. Stáhnout výsledky

ZIP obsahuje tabulky, souhrn a graf.


In [ ]:
from shutil import make_archive

archive = make_archive("vysledky_predikce", "zip", root_dir="predikce")
files.download(archive)
